<div style="background: linear-gradient(135deg, #1e1b4b, #312e81); color:white; padding:25px; border-radius:10px; 
            text-align:center; font-family:'Segoe UI', sans-serif;">

  <h1 style="margin-bottom:8px;"> WikiArt Image Classification</h1>
  <h3 style="margin-top:0; font-style:italic; font-weight:normal; color:#a5b4fc;">
    CNN from Scratch
  </h3>

  <hr style="width:60%; border:1px solid #6366f1; margin:15px auto;">

  <p style="margin:5px 0; font-size:15px;">
    <b>Group Project</b> - Deep Learning (2025/2026)
  </p>
  <p style="margin:0; font-size:13px; color:#c7d2fe;">
    Master in Data Science and Advanced Analytics - Nova Information Management School
  </p>
</div>

<br>

<div style="background-color:#1e293b; color:#e0e7ff; padding:15px 20px; border-left:5px solid #6366f1; 
            border-radius:6px; font-family:'Segoe UI', sans-serif; font-size:14px;">
<b>Notebook Description</b><br>
This notebook presents a comprehensive implementation of a Convolutional Neural Network (CNN) from scratch for the task of image classification on the WikiArt dataset.
</div>

<br>

**<h3>Table of Contents</h3>**
* [1. Environment Setup](#1-environment-setup)
* [2. Model Implementation](#2-model)
* [3. Model Evaluation](#3-model2)

<div id="1-environment-setup" style="background-color:#1e293b; padding:18px; border-radius:6px;">
  <h2 style="margin:0; color:#a5b4fc;">
    1. Setup
  </h2>
</div>

## 1.1 Libraries imports

In [ ]:
import os
import sys
import yaml
import tensorflow as tf
import keras
from keras import layers, regularizers

# Auto-reload local modules while editing this notebook
%load_ext autoreload
%autoreload 2

sys.path.append(os.path.abspath('../src'))

from utils import (
    build_standard_augmentation,
    build_standard_callbacks,
    evaluate_model,
    load_datasets,
    plot_learning_curves,
    prepare_dataset_pipeline,
    save_history,
    set_seeds,
)

# load config
with open('../config.yml', 'r') as f:
    config = yaml.safe_load(f)

SEED = config['seed']
set_seeds(SEED)

<div id="2-model" style="background-color:#1e293b; padding:18px; border-radius:6px;">
  <h2 style="margin:0; color:#a5b4fc;">
    Model Implementation
  </h2>
</div>

In [ ]:
IMG_SIZE = tuple(config['img_size'])
BATCH_SIZE = config['batch_size']
NUM_CLASSES = config['num_classes']

# All paths relative to notebooks/
train_dir = config['paths']['train_dir']
val_dir = config['paths']['val_dir']
test_dir = config['paths']['test_dir']

train_ds, val_ds, test_ds, class_names = load_datasets(
    train_dir, val_dir, test_dir,
    img_size=IMG_SIZE, batch_size=BATCH_SIZE,
)

### Model Architecture - CNN from Scratch

The model is a custom residual CNN built entirely from scratch, without any pretrained weights. Raw RGB images are fed at the configured resolution (`IMG_SIZE`), rescaled from `[0, 255]` to `[0, 1]` inside the model graph.

The core building block is a **residual block**: two 3x3 convolutional layers with Batch Normalization and ReLU, plus a skip connection that adds the block's input directly to its output (with a 1×1 projection if the filter count changes). This allows gradients to flow more easily and lets the network learn residual corrections rather than full transformations.

Four residual stages are stacked with progressively increasing filter counts (**32 -> 64 -> 128 -> 256**), each followed by MaxPooling to halve the spatial dimensions - capturing low-level features (edges, textures) in early stages and higher-level patterns (brushstrokes, compositional style) in later ones. L2 regularization (`1e-4`) is applied to all convolutional layers.

After the final stage, **Global Average Pooling** collapses the spatial maps into a single vector. The classifier head then applies `Dense(256)` with ReLU, `Dropout(0.5)`, and a final `Dense(NUM_CLASSES)` with softmax over the 23 artist classes.

In [ ]:
train_ds, val_ds, test_ds = prepare_dataset_pipeline(
    train_ds, val_ds, test_ds, seed=SEED
)

data_augmentation = build_standard_augmentation()

def residual_block(x, filters):
    shortcut = x
    x = layers.Conv2D(filters, 3, padding="same", kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Conv2D(filters, 3, padding="same", kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    if shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, 1, padding="same")(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)
    x = layers.Add()([x, shortcut])
    x = layers.Activation("relu")(x)
    return x

inputs = keras.Input(shape=(*IMG_SIZE, 3))
x = data_augmentation(inputs)
x = layers.Rescaling(1.0 / 255)(x)

x = residual_block(x, 32)
x = layers.MaxPooling2D()(x)

x = residual_block(x, 64)
x = layers.MaxPooling2D()(x)

x = residual_block(x, 128)
x = layers.MaxPooling2D()(x)

x = residual_block(x, 256)
x = layers.MaxPooling2D()(x)

x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

model = keras.Model(inputs, outputs, name="cnn_residual_v5")

### Training

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy", keras.metrics.F1Score(average="macro", name="f1_macro")],
)

# Save model in models directory (relative to notebooks/)
callbacks = build_standard_callbacks(
    checkpoint_path=config['models']['residual_block']['checkpoint'],
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=config['max_epochs'],
    callbacks=callbacks,
)

### Learning Curves

In [ ]:
plot_learning_curves(history, title="Scratch CNN")

<div id="3-model2" style="background-color:#1e293b; padding:18px; border-radius:6px;">
  <h2 style="margin:0; color:#a5b4fc;">
    3. Model Evaluation
  </h2>
</div>

### Test Evaluation

In [ ]:
metrics_cnn = evaluate_model(model, test_ds, class_names, "Scratch CNN")

In [ ]:
history_path = config['models']['residual_block']['history']
os.makedirs(os.path.dirname(history_path), exist_ok=True)
save_history(history, history_path)